In [ ]:
############################################## SETTINGS ##############################################  
sim = 'sim2277'

# settings
cell_type='ispn'
model = 3

import os, sys
# compute absolute path of main folder
cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

# set CWD or add to path
os.chdir(main_dir)
sys.path.insert(0, main_dir)

# then import/run settings
%run -i settings.py

from analysis_functions import *

# Get the current working directory
current_wd = os.getcwd()
# 'sim' + str(sim)

downsample = True

home = os.path.expanduser('~')

# Set working directory
external = False
external_name = 'MacOS10'
target = f'model{model}'

if not external:
    if downsample:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'
else:
    if downsample:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'


# create path to directory to save analysis
base_path = os.path.join(home, 'Documents', 'Repositories', 'msNEURON_Belal2026', 'analysis')
sim_image_path = os.path.join(base_path, cell_type, sim)
os.makedirs(sim_image_path, exist_ok=True)

save = False

In [ ]:
# Basic plot settings
s = 5 # splprep fits a parametric B-spline to your 2D points 
lwd = 1 # standardise line width

height1 = 600
width1 = 600

height2 = 600
width2 = 800

if cell_type == 'ispn':
    plane='yx'
    mirror=False
    dend_name = 'dend[12]'
    
else:
    plane='xy'
    mirror=False
    dend_name = 'dend[7]'

if downsample:
    ds = 10

In [ ]:
# load simulations
sims_dir = os.path.join(wd, sim)
# sim_data = load_data_dicts(sims_dir, verbose=True)

sim_data = load_data_dicts(wd=wd, sim=sim, cell_type=cell_type, verbose=True)

In [ ]:
# check files loaded in correct order
for name in sim_data.keys():
    print(name)

In [ ]:
report_memory(verbose=True)

In [ ]:
# check if coordinates exist
coords_exist = summarise_cell_coordinates(sim_data)

# if so load from v_all_3D else retrieve from cell_build
# this is because the methods to retrieve the cell_coordinates differ so it is important when plotting 3D heatmaps that the correct coordinates are used
# the method to default will only be used when the simulations have no 3d data 
if coords_exist:
    first_sim = next(iter(sim_data.values()))
    v_all_3D = first_sim['v_all_3D']
    first_key = next(iter(v_all_3D)) 
    
    cell_coordinates = np.array(v_all_3D[first_key]['cell_coordinates'], dtype=object)
    _, _, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
else:
    cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
    cell_coordinates = []
    
    for sec in cell.somalist:
        h('access ' + sec.name())
        x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
        cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])
    
    # Setup for dendritic sections
    for sec in cell.dendlist:
        for seg in sec:
            x, y, z, diam = interpolate_3d(sec, seg.x)
            cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])
        
    cell_coordinates = np.array(cell_coordinates, dtype=object)

    
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=lwd, s=s, color='grey', height=height1, width=width1, plane='xy', mirror=False)

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
fig_morphology.write_image(f"{sim_image_path}/morphology.svg", format='svg', scale=3)


In [ ]:
# Morphology plot with synaptic locations

# extract names and locations of GLUT and GABA inputs from metadata
# using last simulation to get all GABA locations used (fixed)
dend_gaba_all = list(sim_data.values())[-1]['metadata']['dend_gaba']
gaba_locations_all = list(sim_data.values())[-1]['metadata']['gaba_locations']

dend_glut = list(sim_data.values())[-1]['metadata']['dend_glut']
glut_locations = list(sim_data.values())[-1]['metadata']['glutamate_locations']

if len(dend_glut) != len(glut_locations): # one upstate site 
    dend_glut = dend_glut * len(glut_locations)
    
# indices
idxs_gaba = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=dend_gaba_all, target_location=gaba_locations_all)
idxs_glut = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=dend_glut, target_location=glut_locations)

# coordinates
coords_gaba = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=dend_gaba_all, target_location=gaba_locations_all)
coords_glut = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=dend_glut, target_location=glut_locations)

# with interpolation
fig_morphology = morphology_plot(cell_coordinates=cell_coordinates, dend_tree=dend_tree, idxs=[coords_gaba, coords_glut], 
                                 title='', idxs_colors = ['#7cc57a', '#CD5C5C'], dot_size=[4,6], lwd=lwd, s=s, color='grey', 
                                 height=height1, width=width1, plane=plane, mirror=mirror, alpha=[0.5, 0.25])

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology2.svg", format='svg', scale=3)

In [ ]:
compare_synapse_coords("Glutamatergic", dend_glut, glut_locations, coords_glut)

In [ ]:
compare_synapse_coords("GABAergic", dend_gaba_all, gaba_locations_all, coords_gaba)

In [ ]:
# Simple output traces
# downsample plots if original simulation is sampled at high rate eg 0.025 ms
# if simulation was subsequently downsampled then no need to downsample plots

dt = next(iter(sim_data.values()))['metadata']['dt']
sim_time = next(iter(sim_data.values()))['metadata']['sim_time']

if downsample:
    ds_plot = 1
    dt = ds * dt
    
else:
    ds_plot = 10
    
Vsoma = []
Vdend = []

for variables in sim_data.values():
    if 'vsoma' in variables:
        Vsoma.append(variables['vsoma'])
    if 'vdend' in variables:
        Vdend.append(variables['vdend'])    
        
Vsoma_fig = plot3_dt(Vsoma, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='somatic voltage', lwd=lwd,
                     yrange=[-90, -55], yabline = [-86, -60], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=5, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)        

Vdend_fig = plot3_dt(Vdend, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='dendritic voltage', lwd=lwd,
                     yrange=[-86, -20], yabline = [-86, -30], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=10, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)

Vsoma_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
Vdend_fig.show(config={"toImageButtonOptions": {"format": "svg"}})

Vsoma_fig.write_image(f"{sim_image_path}/Vsoma.svg", format='svg', scale=3)
Vdend_fig.write_image(f"{sim_image_path}/Vdend.svg", format='svg', scale=3)


In [ ]:
# animation
Vdend2animate_fig = plot3_dt(Vdend[0:100], yaxis='V (mV)', stim_time = 150, sim_time=sim_time, black_trace=0, gray_trace=1,
                title='somatic voltage', yrange=[-86, -20], xrange=[50, 500], yabline = [-86, -60], 
                width=800, height=400, baseline=100, x_err_bar=25, y_err_bar=10, dt=dt, ds=ds_plot, 
                offset=False, legend=False, alpha=0.6)        

gray_idx_in_fig = len(Vdend2animate_fig.data)-1 
anim_fig = animate_traces(Vdend2animate_fig, skip_indices=[0, gray_idx_in_fig], highlighted_trace=1)
anim_fig.show()

if save:
    export_html(anim_fig, f"{sim_image_path}/Vdend_anim.html")
    export_mp4(anim_fig, f"{sim_image_path}/Vdend_anim.mp4")    

In [ ]:
if downsample:
    # reconstruct 
    timing_range = range(120, 631, 30)
    if sim in ['sim2271', 'sim2273', 'sim2275', 'sim2277', 'sim2279']:
        timing_range = range(120, 631, 3) 
        
    timing_range = np.insert(timing_range, 0, [150, 150])

else:
    timing_range = next(iter(sim_data.values()))['metadata']['timing_range']


timing_range_full = timing_range

timing_range = range(120, 631, 30)

idx = np.where(np.isin(timing_range_full, timing_range))[0]

Vsoma_idx = [Vsoma[i] for i in idx]
Vdend_idx = [Vdend[i] for i in idx]

Vsoma_fig1 = plot3_dt(Vsoma_idx, yaxis='V (mV)', stim_time = 150, sim_time=800, title='somatic voltage', lwd=lwd,
                     yrange=[-90, -55], yabline = [-86, -60], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=5, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)        

Vdend_fig1 = plot3_dt(Vdend_idx, yaxis='V (mV)', stim_time = 150, sim_time=800, title='dendritic voltage', lwd=lwd,
                     yrange=[-86, -20], yabline = [-86, -30], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=10, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)

Vsoma_fig1.show(config={"toImageButtonOptions": {"format": "svg"}})
Vdend_fig1.show(config={"toImageButtonOptions": {"format": "svg"}})

Vsoma_fig1.write_image(f"{sim_image_path}/Vsoma1.svg", format='svg', scale=3)
Vdend_fig1.write_image(f"{sim_image_path}/Vdend1.svg", format='svg', scale=3)



In [ ]:
stim_time = next(iter(sim_data.values()))['metadata']['stim_time']
rel_timing = [t - stim_time for t in timing_range_full[2:]]
start_time  = np.min(timing_range)
baseline = 20

In [ ]:
P2_soma = rel_peak_fun(V=Vsoma, timing_range=timing_range_full, dt=dt, start_time=start_time, baseline=baseline, omit_trace=0, sub_trace=1)[0]
P2_dend = rel_peak_fun(V=Vdend, timing_range=timing_range_full, dt=dt, start_time=start_time, baseline=baseline, omit_trace=0, sub_trace=1)[0]

P3_soma = peak_fun(V=Vsoma, timing_range=timing_range_full, dt=dt, start_time=start_time, baseline=baseline)[0]
P3_dend = peak_fun(V=Vdend, timing_range=timing_range_full, dt=dt, start_time=start_time, baseline=baseline)[0]

P2_P1_soma = np.round([P / P2_soma[0] for P in P2_soma[2:]], 5)
P2_P1_dend = np.round([P / P2_dend[0] for P in P2_dend[2:]], 5)

P3_P1_soma = np.round([P / P3_soma[0] for P in P3_soma[2:]], 5)
P3_P1_dend = np.round([P / P3_dend[0] for P in P3_dend[2:]], 5)


In [ ]:
# xrange=(-40, 482)
xrange=(-40, 362)

plot4(rel_timing, P3_P1_soma, P3_P1_dend, sim_image_path=sim_image_path,
               width=4, height=2.25, marker_size=1, lwd=1, xrange=xrange, yrange=(0, 6), x_tick_step=40,
               x_title = r'$\mathrm{t}_{\mathrm{GLUT}} - \mathrm{t}_{\mathrm{GABA}}\;\mathrm{(ms)}$', y_title='P₃/P₁',
               save_name='P3_P1_plot.svg', save=True)

In [ ]:
plot4(rel_timing, P2_P1_soma, P2_P1_dend, sim_image_path=sim_image_path,
               width=4, height=2.25, marker_size=1, lwd=1, xrange=xrange, yrange=(0, 5), x_tick_step=40, 
               x_title = r'$\mathrm{t}_{\mathrm{GLUT}} - \mathrm{t}_{\mathrm{GABA}}\;\mathrm{(ms)}$', y_title='P₂/P₁',
               save_name='P2_P1_plot.svg', save=True)

In [ ]:
plot4(rel_timing, P2_P1_soma, P2_P1_dend, sim_image_path=sim_image_path,
               width=4, height=2.25, marker_size=1, lwd=1, xrange=xrange, yrange=(0, 2), x_tick_step=40, 
               x_title = r'$\mathrm{t}_{\mathrm{GLUT}} - \mathrm{t}_{\mathrm{GABA}}\;\mathrm{(ms)}$', y_title='P₂/P₁',
               save_name='P2_P1_plot2.svg', save=True)